In [1]:
import mne
import pandas as pd
import numpy as np
import math
import re
from collections import namedtuple
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns
from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.linear_model import Ridge
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import fdrcorrection
from IPython.display import clear_output

In [28]:
DATA_DIR = "/Users/jowanglin/Word-Position-Effect_BLP-lab/Preprocessing/ica-data"
RAW_FILE_PATH = "SUBJ020/subj020.set"
EPOCHS_FILE_PATH = "SUBJ020/subj020_chan_rr_elist_bin_epbaseline_AD_BPdot1-30.set"

In [34]:
def to_num_str(cell):
    if isinstance(cell, str):
        if "boundary" in cell:
            return "-1"
        else:
            return re.findall(r"\d+", cell)[0]
    else:
        return cell

raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{RAW_FILE_PATH}", preload=True, verbose=False)
df_annot = pd.DataFrame(raw.annotations)
df_annot = df_annot.map(to_num_str)
display(df_annot)

/var/folders/n3/nqqvqq1n0jx25dfd3nyxjxmh0000gn/T/ipykernel_79535/53527967.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{RAW_FILE_PATH}", preload=True, verbose=False)


,onset,duration,description,orig_time,extras
0,0.000,0.000,-1,None,{}
1,34.625,0.001,254,None,{}
2,35.209,0.001,1,None,{}
3,45.816,0.001,254,None,{}
4,46.337,0.001,251,None,{}
...,...,...,...,...,...
3223,3239.549,0.001,253,None,{}
3224,3241.064,0.001,1,None,{}
3225,3242.796,0.001,254,None,{}
3226,3244.315,0.001,1,None,{}


In [38]:
def revise_annot(df_annot: pd.DataFrame, *,
                 item_codes: list, condition_codes: list, non_final: str) -> mne.Annotations:
    desc = list(df_annot["description"])
    desc_revised = desc.copy()

    for i, d in enumerate(desc):
        if d in item_codes:
            if desc[i+1] == non_final:
                step = 0
        elif d == non_final:
            step += 1
            desc_revised[i] = "w" + str(step)
        
        elif d in condition_codes:
            desc_revised[i-step: i] = [s + "/" + d for s in desc_revised[i-step: i]]

    annot = mne.Annotations(onset=df_annot["onset"], duration=df_annot["duration"], description=desc_revised)
    return annot
            
item_codes = list(df_annot["description"].value_counts()[df_annot["description"].value_counts() == 2].index)
item_codes.append("1")
condition_codes = list(df_annot["description"].value_counts()[df_annot["description"].value_counts() == 28].index)
non_final = "255"

revised_annot = revise_annot(df_annot, item_codes=item_codes, condition_codes=condition_codes, non_final=non_final)
raw.set_annotations(revised_annot)
df_revised_annot = pd.DataFrame(raw.annotations)
display(df_revised_annot)

,onset,duration,description,orig_time,extras
0,0.000,0.000,-1,None,{}
1,34.625,0.001,254,None,{}
2,35.209,0.001,1,None,{}
3,45.816,0.001,254,None,{}
4,46.337,0.001,251,None,{}
...,...,...,...,...,...
3223,3239.549,0.001,253,None,{}
3224,3241.064,0.001,1,None,{}
3225,3242.796,0.001,254,None,{}
3226,3244.315,0.001,1,None,{}


In [43]:
pd.DataFrame(revised_annot).iloc[3:23]

,onset,duration,description,orig_time,extras
3,45.816,0.001,254,None,{}
4,46.337,0.001,251,None,{}
5,47.354,0.001,105,None,{}
6,47.871,0.001,w1/244,None,{}
7,48.404,0.001,w2/244,None,{}
8,48.921,0.001,w3/244,None,{}
9,49.437,0.001,w4/244,None,{}
10,49.954,0.001,w5/244,None,{}
11,50.471,0.001,w6/244,None,{}
12,50.987,0.001,w7/244,None,{}
